# 01. 8-class YOLO11n 검증 및 런타임 반영

이 노트북은 Colab에서 중간 중단한 8-class YOLO11n 산출물을 확인한다.

목표:

- `best.pt`, `best.onnx`, `results.csv` 존재 확인
- 클래스 순서가 런타임 config와 같은지 확인
- 학습 중단 시점까지의 validation metric 확인
- ONNX Runtime에서 모델 입출력 shape smoke test
- 최종 패키지 `40_drive_runtime/team3_final_drive/models/sign/`에 ONNX 복사

주의:

- 이 노트북은 재학습을 하지 않는다.
- 실주행 event threshold는 `40_drive_runtime/team3_final_drive/config.py`에서 조정한다.

In [1]:
from pathlib import Path
import json
import shutil

import pandas as pd

PROJECT = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization")
RUN_DIR = Path(r"~\Downloads\runs_8class_yolo11n\sign_traffic_8class_yolo11n_v1")
WEIGHTS = RUN_DIR / "weights"
BEST_PT = WEIGHTS / "best.pt"
BEST_ONNX = WEIGHTS / "best.onnx"
RESULTS_CSV = RUN_DIR / "results.csv"
ARGS_YAML = RUN_DIR / "args.yaml"

OUT_DIR = PROJECT / "10_experiments" / "22_yolo_sign_traffic_8class_train" / "review_outputs" / "01_verify_8class_yolo_and_install_runtime"
SHARED_MODEL_DIR = PROJECT / "20_shared_assets" / "models" / "sign_traffic_8class_yolo11n"
PKG_SIGN_DIR = PROJECT / "40_drive_runtime" / "team3_final_drive" / "models" / "sign"

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_DIR:", RUN_DIR)
for p in [BEST_PT, BEST_ONNX, RESULTS_CSV, ARGS_YAML]:
    print(p.name, p.exists(), p.stat().st_size if p.exists() else None)

RUN_DIR: ~\Downloads\runs_8class_yolo11n\sign_traffic_8class_yolo11n_v1
best.pt True 15933345
best.onnx True 10609588
results.csv True 5972
args.yaml True 1925


## 1. 클래스 순서 확인

런타임은 YOLO 출력 channel index를 class name으로 해석한다.
따라서 아래 순서가 `config.py`의 `SIGN_MODEL["classes"]`와 반드시 같아야 한다.

In [2]:
from ultralytics import YOLO

model = YOLO(str(BEST_PT))
names = model.names
expected = {0: "green", 1: "horn", 2: "left", 3: "red", 4: "right", 5: "speed_20", 6: "stop", 7: "straight"}

print("model.names:", names)
print("expected:", expected)
assert names == expected, "Class order mismatch. Runtime config.py must be updated before deployment."

model.names: {0: 'green', 1: 'horn', 2: 'left', 3: 'red', 4: 'right', 5: 'speed_20', 6: 'stop', 7: 'straight'}
expected: {0: 'green', 1: 'horn', 2: 'left', 3: 'red', 4: 'right', 5: 'speed_20', 6: 'stop', 7: 'straight'}


## 2. 학습 결과 요약

중간 중단했기 때문에 epoch 80까지 간 결과는 아니다.
하지만 `best.pt`는 validation metric 기준 최고 checkpoint이며, 지금은 이걸 runtime 후보로 사용한다.

In [3]:
df = pd.read_csv(RESULTS_CSV)
df.columns = [c.strip() for c in df.columns]
metric_cols = ["epoch", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]
display(df[metric_cols].tail(10))

best_idx = df["metrics/mAP50-95(B)"].idxmax()
best_row = df.loc[best_idx, metric_cols]
print("best row by mAP50-95:")
display(best_row.to_frame().T)

,epoch,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B)
24,25,0.97762,0.98227,0.98030,0.67180
25,26,0.97043,0.97452,0.97878,0.65938
26,27,0.96685,0.97873,0.98635,0.66778
27,28,0.97162,0.97305,0.98309,0.68119
28,29,0.97085,0.97223,0.98250,0.67549
29,30,0.96809,0.97419,0.98358,0.67660
30,31,0.97465,0.97903,0.98699,0.68209
31,32,0.97877,0.97454,0.98626,0.68297
32,33,0.97511,0.97806,0.98610,0.67716
33,34,0.97201,0.98288,0.98806,0.68166


best row by mAP50-95:


,epoch,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B)
31,32.0,0.97877,0.97454,0.98626,0.68297


## 3. ONNX Runtime smoke test

여기서는 정확도 평가가 아니라 ONNX 파일이 실제로 로드되고, YOLO11n detect head 형태의 출력을 내는지만 확인한다.

In [4]:
import cv2
import numpy as np
import onnxruntime as ort

session = ort.InferenceSession(str(BEST_ONNX), providers=["CPUExecutionProvider"])
inp = session.get_inputs()[0]
out = session.get_outputs()[0]
print("input:", inp.name, inp.shape, inp.type)
print("output:", out.name, out.shape, out.type)

dummy = np.zeros((1, 3, 640, 640), dtype=np.float32)
pred = session.run([out.name], {inp.name: dummy})[0]
print("pred shape:", pred.shape)
assert pred.shape[1] == 12 or pred.shape[2] == 12, "Expected YOLO output channel 4+8=12."

input: images [1, 3, 640, 640] tensor(float)
output: output0 [1, 12, 8400] tensor(float)
pred shape: (1, 12, 8400)


## 4. 공유 모델 폴더와 최종 패키지에 복사

복사 대상:

- 공유 보관: `20_shared_assets/models/sign_traffic_8class_yolo11n/`
- 최종 런타임: `40_drive_runtime/team3_final_drive/models/sign/sign_traffic_8class_yolo11n_best.onnx`

In [5]:
def copy2(src, dst):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print("copied:", src, "->", dst)

copy2(BEST_ONNX, SHARED_MODEL_DIR / "sign_traffic_8class_yolo11n_best.onnx")
copy2(BEST_PT, SHARED_MODEL_DIR / "sign_traffic_8class_yolo11n_best.pt")
copy2(RESULTS_CSV, SHARED_MODEL_DIR / "training_results.csv")
copy2(ARGS_YAML, SHARED_MODEL_DIR / "train_args.yaml")
copy2(BEST_ONNX, PKG_SIGN_DIR / "sign_traffic_8class_yolo11n_best.onnx")

summary = {
    "source_run_dir": str(RUN_DIR),
    "class_names": names,
    "best_epoch_by_map50_95": int(best_row["epoch"]),
    "best_map50_95": float(best_row["metrics/mAP50-95(B)"]),
    "best_map50": float(best_row["metrics/mAP50(B)"]),
    "runtime_onnx": str(PKG_SIGN_DIR / "sign_traffic_8class_yolo11n_best.onnx"),
}
(OUT_DIR / "verify_8class_yolo_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
summary

copied: ~\Downloads\runs_8class_yolo11n\sign_traffic_8class_yolo11n_v1\weights\best.onnx -> ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\20_shared_assets\models\sign_traffic_8class_yolo11n\sign_traffic_8class_yolo11n_best.onnx
copied: ~\Downloads\runs_8class_yolo11n\sign_traffic_8class_yolo11n_v1\weights\best.pt -> ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\20_shared_assets\models\sign_traffic_8class_yolo11n\sign_traffic_8class_yolo11n_best.pt
copied: ~\Downloads\runs_8class_yolo11n\sign_traffic_8class_yolo11n_v1\results.csv -> ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\20_shared_assets\models\sign_traffic_8class_yolo11n\training_results.csv
copied: ~\Downloads\runs_8class_yolo11n\sign_traffic_8class_yolo11n_v1\args.yaml -> ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\20_shared_assets\models\sign_traffic_8class_yolo11n\train_args.yaml
copied: ~\Downloads\runs_8class_yolo11n\sign_traffic_8class_y

{'source_run_dir': '~\\Downloads\\runs_8class_yolo11n\\sign_traffic_8class_yolo11n_v1',
 'class_names': {0: 'green',
  1: 'horn',
  2: 'left',
  3: 'red',
  4: 'right',
  5: 'speed_20',
  6: 'stop',
  7: 'straight'},
 'best_epoch_by_map50_95': 32,
 'best_map50_95': 0.68297,
 'best_map50': 0.98626,
 'runtime_onnx': '~\\02_Projects\\University\\26-1_EmbeddedArtificialSystemOptimization\\40_drive_runtime\\team3_final_drive\\models\\sign\\sign_traffic_8class_yolo11n_best.onnx'}

## 5. 다음 확인

이후 Pi에서는 다음 순서로 확인한다.

1. `python main_log.py`
   - 이벤트 로그가 터미널에 뜨는지 확인
2. `python main_overlay.py`
   - bbox와 이벤트 후보 위치 확인
3. `python main.py`
   - 최종 시연용 무오버헤드 실행

실주행에서 조정할 값은 `config.py`의 `SIGN_EVENT`, `STATE`, `RUNTIME`이다.